In [4]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate_hkqai").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"new_dataset/{data_set}.json") as f:
    json_data = json.load(f)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict_ccpvdz"]
    # full_subset_dict = json.load(f)["full_subset_dict_test"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for data_path in data_path_list:
        data = pd.read_csv(data_path)
        data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data["dft_ene"].to_numpy() * 627.5094733748099
        data_scf = data["scf_ene"].to_numpy() * 627.5094733748099
        data_cc = data["cc_ene"].to_numpy() * 627.5094733748099

        data_error_scf_ele = data["error_scf_ele"].to_numpy()
        data_error_dft_ele = data["error_dft_ele"].to_numpy()
        data_error_scf_dip = data["error_scf_dip"].to_numpy()
        data_error_dft_dip = data["error_dft_dip"].to_numpy()
        data_subset[f"{data_path_name}_summary"] = {
            "error_scf_ele": data_error_scf_ele,
            "error_dft_ele": data_error_dft_ele,
            "error_scf_dip": data_error_scf_dip,
            "error_dft_dip": data_error_dft_dip,
        }

        if "delta_d3bj" in data.columns:
            data_d3bj = data["delta_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_d3bj = np.zeros_like(data_dft)
        if "delta_d3zero" in data.columns:
            data_d3zero = data["delta_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_d3zero = np.zeros_like(data_dft)

        if "modified_dft_d3bj" in data.columns:
            data_dft_d3bj = data["modified_dft_d3bj"].to_numpy() * 627.5094733748099
            # data_dft_d3bj = data_d3bj
        else:
            data_dft_d3bj = data_d3bj
        if "modified_dft_d3zero" in data.columns:
            data_dft_d3zero = data["modified_dft_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3zero = data_d3zero

        if "modified_ai_d3bj" in data.columns:
            data_ai_d3bj = data["modified_ai_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3bj = data_d3bj
        if "modified_ai_d3zero" in data.columns:
            data_ai_d3zero = data["modified_ai_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3zero = data_d3zero

        del data_d3bj, data_d3zero

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "dft_d3bj": [],
                "dft_d3zero": [],
                "ai": [],
                "ai_d3bj": [],
                "ai_d3zero": [],
                "cc": [],
                "error_scf_ele": [],
                "error_dft_ele": [],
                "error_scf_dip": [],
                "error_dft_dip": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    if verbose > 0:
                        print(
                            f"Warning: {i_molecule_name} not found in {data_path.stem} data file"
                        )
                    continue
                data_subset[name_subset]["error_scf_ele"].append(
                    data_error_scf_ele[col[0]]
                )
                data_subset[name_subset]["error_dft_ele"].append(
                    data_error_dft_ele[col[0]]
                )
                data_subset[name_subset]["error_scf_dip"].append(
                    data_error_scf_dip[col[0]]
                )
                data_subset[name_subset]["error_dft_dip"].append(
                    data_error_dft_dip[col[0]]
                )

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_dft_d3bj = 0
                atomic_energy_dft_d3zero = 0
                atomic_energy_ai = 0
                atomic_energy_ai_d3bj = 0
                atomic_energy_ai_d3zero = 0
                atomic_energy_cc = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                        atomic_energy_dft_d3bj += data_dft_d3bj[col[0]] * stoichiometry
                        atomic_energy_dft_d3zero += (
                            data_dft_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_ai += data_scf[col[0]] * stoichiometry
                        atomic_energy_ai_d3bj += data_ai_d3bj[col[0]] * stoichiometry
                        atomic_energy_ai_d3zero += (
                            data_ai_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_cc += data_cc[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["dft_d3bj"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3bj
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["dft_d3zero"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["ai"].append(
                        abs(atomic_energy_ai - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3bj"].append(
                        abs(atomic_energy_ai + atomic_energy_ai_d3bj - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3zero"].append(
                        abs(
                            atomic_energy_ai
                            + atomic_energy_ai_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_ai = np.argsort(data_subset[name_subset]["ai"])[
                    ::-1
                ][:5]
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ][:5]
                for i in range(len(argsort_atomic_energy_ai)):
                    i_reaction_name = data_subset[name_subset]["name"][
                        argsort_atomic_energy_ai[i]
                    ]
                    i_reaction = json_data[f"reaction-{i_subset}"][i_reaction_name]
                    print(
                        f"Top {i+1} AI: {data_subset[name_subset]['ai'][argsort_atomic_energy_ai[i]]} kcal/mol, {i_reaction_name} in {name_subset}",
                    )
                    systems_list = i_reaction["systems"]
                    stoichiometry_list = i_reaction["stoichiometry"]
                    for j in range(len(systems_list)):
                        mole_name = (
                            systems_list[j]
                            if i_subset == "BH76RC"
                            else f"{i_subset}-{systems_list[j]}"
                        )
                        stoichiometry = int(stoichiometry_list[j])
                        if mole_name in json_data:
                            if isinstance(json_data[mole_name], str):
                                mole_name = json_data[mole_name]
                        print(f"  {stoichiometry} * {mole_name}", end="")
                        col = np.where(data_name == mole_name)[0]
                        if col.size == 1:
                            error_energy_ai = data_scf[col[0]] - data_cc[col[0]]
                            print(f"  {stoichiometry} * {error_energy_ai}", end="")
                    print()

                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

data_path_name_list = [
    data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for data_path in data_path_list
]
header = pd.MultiIndex.from_product(
    [
        data_path_name_list,
        [
            "AI",
            "DFT",
            "AI_D3BJ",
            "DFT_D3BJ",
            "AI_D3ZERO",
            "DFT_D3ZERO",
            "Processed",
        ],
    ],
    names=["data_path", "Disp type"],
)

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

df_summary_subset_ele = pd.DataFrame(
    columns=pd.MultiIndex.from_product(
        [
            data_path_name_list,
            [
                "error_scf_ele",
                "error_dft_ele",
                "error_scf_dip",
                "error_dft_dip",
            ],
        ],
        names=["data_path", "Ele type"],
    )
)

for data_path in data_path_list:
    mean_absolute_deviation_list = []
    data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_dip"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_dip"]
    )

    for name_set, subset_list_ in full_subset_dict.items():
        subset_ai = {}
        wtmad_1_ai = {}
        wtmad_2_ai = {}
        subset_dft = {}
        wtmad_1_dft = {}
        wtmad_2_dft = {}

        for d3_name in ["", "_d3bj", "_d3zero"]:
            subset_ai[d3_name] = []
            wtmad_1_ai[d3_name] = []
            wtmad_2_ai[d3_name] = []
            subset_dft[d3_name] = []
            wtmad_1_dft[d3_name] = []
            wtmad_2_dft[d3_name] = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_ele")] = (
                np.mean(data_subset[name_subset]["error_scf_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_ele")] = (
                np.mean(data_subset[name_subset]["error_dft_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_dip")] = (
                np.mean(data_subset[name_subset]["error_scf_dip"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_dip")] = (
                np.mean(data_subset[name_subset]["error_dft_dip"])
            )

            if len(data_subset[name_subset]["ai"]) == 0:
                for col_name in [
                    "AI",
                    "DFT",
                    "AI_D3BJ",
                    "DFT_D3BJ",
                    "AI_D3ZERO",
                    "DFT_D3ZERO",
                ]:
                    df_summary_subset.loc[i_subset, (data_path_name, col_name)] = 0
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                for d3_name in ["", "_d3bj", "_d3zero"]:
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"AI{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"ai{d3_name}"])
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"DFT{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"dft{d3_name}"])
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    "DONE"
                    if (
                        (
                            len(data_subset[name_subset]["ai"])
                            == len(data_subset[name_subset]["name"])
                        )
                        and (len(data_subset[name_subset]["ai"]) != 0)
                    )
                    else f"{len(data_subset[name_subset]['ai'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                for d3_name in ["", "_d3bj", "_d3zero"]:
                    subset_ai[d3_name] = np.append(
                        subset_ai[d3_name], data_subset[name_subset][f"ai{d3_name}"]
                    )
                    subset_dft[d3_name] = np.append(
                        subset_dft[d3_name], data_subset[name_subset][f"dft{d3_name}"]
                    )
                    wtmad_1_ai[d3_name] = np.append(
                        wtmad_1_ai[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"ai{d3_name}"]),
                    )
                    wtmad_1_dft[d3_name] = np.append(
                        wtmad_1_dft[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"dft{d3_name}"]),
                    )
                    wtmad_2_ai[d3_name] = np.append(
                        wtmad_2_ai[d3_name],
                        data_subset[name_subset][f"ai{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                    wtmad_2_dft[d3_name] = np.append(
                        wtmad_2_dft[d3_name],
                        data_subset[name_subset][f"dft{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["ai"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)
        for d3_name in ["", "_d3bj", "_d3zero"]:
            mean_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(subset_ai[d3_name])
            )
            mean_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(subset_dft[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(wtmad_1_ai[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(wtmad_1_dft[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.sum(wtmad_2_ai[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.sum(wtmad_2_dft[d3_name])
            )
        mean_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(
        mean_absolute_deviation_list
    )
    print(
        f"Mean absolute deviation for {data_path_name}: {mean_absolute_deviation:.4f} kcal/mol"
    )
    for name_set in full_subset_dict.keys():
        for d3_name in ["", "_d3bj", "_d3zero"]:
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[
                    name_set, (data_path_name, f"DFT{d3_name.upper()}")
                ]
            )

# print("Summary")
# display(df_summary_subset_ele)
print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# # save summary to csv with date
# df_summary_subset_ele.to_csv(f"../validate/df_summary_subset_ele_{date}.csv")
# df_summary_subset.to_csv(f"../validate/summary_subset_{date}.csv")
# mean_subset.to_csv(f"../validate/mean_subset_{date}.csv")
# wtmad_1_subset.to_csv(f"../validate/wtmad_1_subset_{date}.csv")
# wtmad_2_subset.to_csv(f"../validate/wtmad_2_subset_{date}.csv")
# # save summary to excel with date
# df_summary_subset_ele.to_excel(f"../validate/df_summary_subset_ele_{date}.xlsx")
# df_summary_subset.to_excel(f"../validate/summary_subset_{date}.xlsx")
# mean_subset.to_excel(f"../validate/mean_subset_{date}.xlsx")
# wtmad_1_subset.to_excel(f"../validate/wtmad_1_subset_{date}.xlsx")
# wtmad_2_subset.to_excel(f"../validate/wtmad_2_subset_{date}.xlsx")

cc-pVDZ
Top 1 AI: 32.46840298047755 kcal/mol, 131 in 4190857_W4_11
  -1 * W4_11-oclo  -1 * -31.239180416159797  2 * W4_11-o  2 * 0.5579400721035199  1 * W4_11-cl  1 * 0.11334242013981566
Top 2 AI: 32.30097158405988 kcal/mol, 136 in 4190857_W4_11
  -1 * W4_11-foof  -1 * -32.417518583271885  2 * W4_11-f  2 * -0.6162135717095225  2 * W4_11-o  2 * 0.5579400721035199
Top 3 AI: 30.709564340068027 kcal/mol, 128 in 4190857_W4_11
  -1 * W4_11-s4-c2v  -1 * -30.55099387851078  4 * W4_11-s  4 * 0.039642615389311686
Top 4 AI: 25.991324992195587 kcal/mol, 134 in 4190857_W4_11
  -1 * W4_11-fo2  -1 * -25.491658419690793  1 * W4_11-f  1 * -0.6162135717095225  2 * W4_11-o  2 * 0.5579400721035199
Top 5 AI: 25.247065533243585 kcal/mol, 107 in 4190857_W4_11
  -1 * W4_11-so3  -1 * -23.533602701558266  1 * W4_11-s  1 * 0.039642615389311686  3 * W4_11-o  3 * 0.5579400721035199
Top 1 DFT: 64.94206620124169 kcal/mol
Top 2 DFT: 64.82773676099896 kcal/mol
Top 3 DFT: 58.78632347400708 kcal/mol
Top 4 DFT: 55.623948

/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packa

data_path   4190857                                                       \
Disp type        AI        DFT   AI_D3BJ   DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       5.878346  13.710809  6.034077  14.259652  5.683363  13.713748   
sub2       8.342449   6.615367  7.457728   9.134314  6.206933   6.763238   
sub3       4.165642   6.479422  5.077548   7.660855  4.675188    7.23621   
sub4       0.593172   0.603165  0.692358   1.091271  0.742036   1.152258   
sub5       0.569091   0.636862  0.360009   0.633654  0.335672   0.558308   

data_path             1513512                                             \
Disp type Processed        AI        DFT    AI_D3BJ   DFT_D3BJ AI_D3ZERO   
sub1           DONE  6.282354  13.717062   6.868466  14.265974  6.365582   
sub2           DONE  7.536765   6.612828  11.817093   9.131704  8.930029   
sub3          6 / 7  4.620451    6.26131   5.733397   7.356016  5.317836   
sub4         1 / 11   2.38953   3.272965   3.818374   5.029019  3.790518   
sub5          0 / 8  1.250183   1.322771   0.717525   0.928201  0.665034   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1       13.720054      DONE  
sub2        6.760778      DONE  
sub3        6.935442      DONE  
sub4        4.986451      DONE  
sub5        0.872943      DONE

wtmad_1


data_path    4190857                                                       \
Disp type         AI        DFT   AI_D3BJ   DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1        6.332236    7.42093  5.740168   7.414586  5.713986   7.102103   
sub2       12.258863  12.533853  9.365034   9.970671  9.645465  10.008789   
sub3        4.962526   7.030705  5.831236   8.213003  5.283095   7.607156   
sub4        9.812932  10.226723  9.090578  13.666617  9.740813  14.543333   
sub5        5.690906   6.368619  3.600089   6.336543  3.356719    5.58308   

data_path              1513512                                              \
Disp type Processed         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO   
sub1           DONE   5.917804   7.427913   6.173631   7.422353   5.786012   
sub2           DONE   9.677109  12.530346   7.731942   9.967392   7.407743   
sub3          6 / 7   5.362819   6.982875   6.557344   8.149361   5.935941   
sub4         1 / 11  10.472747  11.046974  12.507724  14.856703  11.478243   
sub5          0 / 8  10.880025  11.599184   6.460544   8.076713   5.881129   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1        7.109349      DONE  
sub2       10.004803      DONE  
sub3        7.519994      DONE  
sub4       13.779798      DONE  
sub5        7.508553      DONE

wtmad_2


data_path   4190857                                                     \
Disp type        AI       DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       5.588196     7.796  5.300608  7.932503  5.245825   7.697743   
sub2       8.226032  7.020342  5.555905  4.588946  6.038396   4.918138   
sub3       3.284345  4.956945  3.941448  5.800699  3.638943   5.489665   
sub4       2.331842  2.017124  3.759936  4.662662   4.13512   5.051146   
sub5       1.607189  1.798584  1.016714  1.789526  0.947983   1.576738   

data_path             1513512                                          \
Disp type Processed        AI       DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO   
sub1           DONE  2.668675  4.045306  2.808633  4.115977  2.675278   
sub2           DONE  3.206768  3.637148  2.047904  2.377001  2.176739   
sub3          6 / 7  1.989474  2.610843   2.42163  3.039958  2.256968   
sub4         1 / 11  4.130245   4.16366  6.081695  6.850967  5.851345   
sub5          0 / 8  4.368745  4.670544  2.805652   3.50597  2.496356   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1        3.994343      DONE  
sub2        2.547486      DONE  
sub3        2.873547      DONE  
sub4        6.614613      DONE  
sub5        3.192324      DONE

Summary of Subset
MAE


data_path    4190857                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
W4_11       7.550798  29.506665   8.934006  31.211555     7.7959  29.860857   
G21EA       1.711269   9.754663   1.708564   9.751886   1.717819   9.757687   
G21IP       2.550915   8.953296   2.557253    8.94666   2.554465   8.951861   
DIPCS10     6.273613  12.311131   6.258881  12.334377   6.306532  12.264933   
PA26        3.605722   2.200766   4.200466   1.900152   3.962178    2.00824   
SIE4x4     20.173028  21.908516  20.602325  22.337814  20.604439  22.339928   
ALKBDE10    8.902005  18.125246   9.590608  18.838948   8.912277  18.135588   
YBDE18      5.933368   8.145393   4.569985   7.265227   4.919947   7.472761   
AL2X6       6.181334   5.659145   1.831383   1.332003    2.43879   1.829895   
HEAVYSB11   4.749555   5.391476   5.679351   8.010474   3.718193   5.984154   
NBPRC         4.9164    2.23255   3.013631   2.544088   3.726127   2.274234   
ALK8        6.616179   4.400063   2.646402   3.174803   3.967115   2.559066   
RC21        4.100054   4.820926   5.069438   6.671006   4.872813     6.0297   
G2RC        6.128987   5.917203   6.551209   6.933515   6.263019   6.417298   
BH76RC      2.449106   3.484561   2.333343   3.539828    2.36072   3.536982   
FH51        4.177508   3.703657   3.122524    3.39478   3.165912   3.256908   
TAUT15       2.45399    2.16453   2.491192   2.175485   2.460292   2.145937   
DC13       11.403446  13.090861    9.49728  12.474047   9.519743  11.475687   
MB16_43     18.68265  15.457354  23.743491   36.86956  15.187126  23.172471   
DARC       13.348619   10.75415   5.627689    3.43411   8.172439    5.57797   
RSE43       2.396741     3.1576   2.156379   2.916276    1.99471    2.74911   
BSR36      14.514193   8.424939    7.16088   1.230819    9.19233   3.103076   
CDIE20      1.421845   1.599507   1.154213   1.336544   1.165153   1.347484   
ISO34       2.246307   2.001743   1.818504   1.577125   1.935109   1.647042   
PArel        1.56488   1.743933   1.546789   1.733749   1.510103   1.645025   
BH76        5.674759   9.107946   6.451724     9.8892   6.240629   9.676053   
BHPERI      2.336375   3.268992    5.13515   7.804724   3.744325     6.4139   
BHDIV10     3.717542   6.277706   4.311655   7.448955    3.82634   6.535109   
INV24        3.13698   3.017328   3.019183   2.781632   3.039437   2.825669   
BHROT27     0.853725   0.853379   0.859734   0.858852   0.784117    0.78352   
PX13         6.74376  11.042023   7.527748   11.82601   6.921594  11.219857   
WCPT18      4.591009   7.967151   5.775844   9.151986   5.368162   8.744304   
RG18        0.282955   0.230018   0.528977   0.655714   0.582816   0.709552   
ADIM6       1.777713   1.779422   1.380732    1.33929   1.522417   1.520708   
S22         0.883211   1.058576   0.817464   2.104982   0.817012    2.13274   
S66                0          0          0          0          0          0   
WATER27            0          0          0          0          0          0   
CARBHB12           0          0          0          0          0          0   
PNICO23            0          0          0          0          0          0   
HAL59              0          0          0          0          0          0   
AHB21              0          0          0          0          0          0   
CHB6               0          0          0          0          0          0   
IL16               0          0          0          0          0          0   
IDISP              0          0          0          0          0          0   
ICONF              0          0          0          0          0          0   
ACONF              0          0          0          0          0          0   
Amino20x4   0.569091   0.636862   0.360009   0.633654   0.335672   0.558308   
PCONF21            0          0          0          0          0          0   
MCONF              0          0          0        